# Analisis Sentimen Publik Terhadap Redenominasi Rupiah
### Studi Kasus: Kebijakan Menteri Keuangan Purbaya Yudhi Sadewa pada Platform YouTube
**Tim Peneliti:**
- Kartika Nur Savira (24031554049)
- Sabrina Putri Amalia (24031554055)
- Variesa Sabita Mumtaza Habib (24031554083)

**Program Studi Sains Data — FMIPA Universitas Negeri Surabaya (UNESA)**

---
### 📌 Daftar Isi Alur Analisis:
1. **Pengumpulan & Pemuatan Data** (YouTube Data API / Dataset CSV)
2. **Text Preprocessing** (Cleaning, Font Normalization, Emoji Removal, Slang Normalization, Stopword Removal)
3. **Pelabelan Sentimen Berbasis Leksikon** (Positif, Netral, Negatif)
4. **Analisis Aspek Isu Kebijakan Moneter** (ABSA 4 Pilar)
5. **Pembagian Data & Ekstraksi Fitur TF-IDF** (N-gram 1-2, 3000 Fitur)
6. **Pelatihan & Komparasi Model Machine Learning** (Naive Bayes vs Linear SVM vs Logistic Regression)
7. **Evaluasi & Visualisasi Matriks Kebingungan (Confusion Matrix)**
8. **Visualisasi Word Cloud & N-gram Terpopuler**
9. **Live Sentiment Predictor** (Pengujian Teks Baru Interaktif)
10. **Ringkasan Eksekutif & Rekomendasi Kebijakan**

## STEP 1: IMPORT LIBRARY & LOAD DATASET

In [ ]:
import os
import re
import csv
import string
import unicodedata
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual theme
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 6)

# Cek file dataset
dataset_file = "youtube_comments.csv" if os.path.exists("youtube_comments.csv") else "data/youtube_comments.csv"
df_raw = pd.read_csv(dataset_file)
print(f"✅ Berhasil memuat {len(df_raw)} baris komentar YouTube.")
df_raw.head()

## STEP 2: PREPROCESSING TEKS & NORMALISASI BAHASA GAUL

In [ ]:
from src.dictionaries import normalisasi_dict, stopwords_indonesia, positive_phrases, negative_phrases, positive_words, negative_words
from src.preprocessor import preprocess_text, clean_for_wordcloud

# Terapkan pembersihan
df_clean = df_raw.copy()
df_clean["clean_comment"] = df_clean["comment"].apply(preprocess_text)
df_clean = df_clean[df_clean["clean_comment"].str.strip() != ""].copy()
print(f"✅ Preprocessing selesai: {len(df_clean)} baris teks valid (setelah membuang baris kosong/noise).")
df_clean[["comment", "clean_comment"]].head()

## STEP 3: PELABELAN SENTIMEN DENGAN RULE-BASED LEXICON SCORING

In [ ]:
from src.lexicon_labeler import get_sentiment_score, get_sentiment_label

df_clean["sentimen"] = df_clean["clean_comment"].apply(get_sentiment_score)
df_clean["label"] = df_clean["sentimen"].apply(get_sentiment_label)

dist = df_clean["label"].value_counts()
pct = df_clean["label"].value_counts(normalize=True) * 100
print("=" * 50)
print("DISTRIBUSI SENTIMEN HASIL LEKSIKON:")
print("=" * 50)
for label in ["Netral", "Positif", "Negatif"]:
    print(f"{label:8s}: {dist.get(label, 0):4d} ({pct.get(label, 0):.1f}%)")

## STEP 4: ANALISIS 4 PILAR ASPEK KEBIJAKAN MONETER (ABSA)

In [ ]:
from src.aspect_analyzer import detect_aspects, summarize_aspects

df_clean["aspects"] = df_clean["clean_comment"].apply(detect_aspects)
aspect_summary = summarize_aspects(df_clean.to_dict(orient="records"))

print("=" * 60)
print("RINGKASAN SENTIMEN BERDASARKAN 4 PILAR KEBIJAKAN:")
print("=" * 60)
for asp, vals in aspect_summary.items():
    if asp != "Umum / Lainnya":
        print(f"• {asp:30s} | Total: {vals["total"]:3d} (Pos: {vals["positif"]}, Net: {vals["netral"]}, Neg: {vals["negatif"]})")

## STEP 5: PEMBAGIAN DATA (TRAIN-TEST SPLIT) & EKSTRAKSI TF-IDF

In [ ]:
try:
    from sklearn.model_selection import train_test_split
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.svm import LinearSVC
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    USE_SKLEARN = True
except ImportError:
    USE_SKLEARN = False

X = df_clean["clean_comment"]
y = df_clean["sentimen"]

if USE_SKLEARN:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)
    print(f"✅ Scikit-Learn TF-IDF: Train {X_train_tfidf.shape} | Test {X_test_tfidf.shape}")
else:
    from src.ml_models import PureTfidfVectorizer
    split_idx = int(len(X) * 0.8)
    X_train, X_test = list(X)[:split_idx], list(X)[split_idx:]
    y_train, y_test = list(y)[:split_idx], list(y)[split_idx:]
    tfidf = PureTfidfVectorizer(max_features=3000, ngram_range=(1, 2))
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)
    print(f"✅ Pure Python TF-IDF: Train {len(X_train)} | Test {len(X_test)}")

## STEP 6: PELATIHAN DAN BENCHMARK MODEL MACHINE LEARNING

In [ ]:
if USE_SKLEARN:
    models = {
        "Naive Bayes": MultinomialNB(),
        "Linear SVM": LinearSVC(max_iter=2000, random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
    }
    results = {}
    for name, m in models.items():
        m.fit(X_train_tfidf, y_train)
        preds = m.predict(X_test_tfidf)
        acc = accuracy_score(y_test, preds)
        results[name] = {"model": m, "acc": acc, "preds": preds}
        print(f"• {name:20s} Akurasi: {acc * 100:.2f}%")
else:
    from src.ml_models import PureMultinomialNB, PureLinearSVM, calculate_metrics
    num_feats = len(tfidf.vocabulary_)
    nb = PureMultinomialNB().fit(X_train_tfidf, y_train, num_feats)
    svm = PureLinearSVM().fit(X_train_tfidf, y_train, num_feats)
    preds_nb = nb.predict(X_test_tfidf)
    preds_svm = svm.predict(X_test_tfidf)
    results = {
        "Naive Bayes": {"acc": calculate_metrics(y_test, preds_nb)["accuracy"], "preds": preds_nb},
        "Linear SVM": {"acc": calculate_metrics(y_test, preds_svm)["accuracy"], "preds": preds_svm}
    }
    for name, res in results.items():
        print(f"• {name:20s} Akurasi: {res["acc"] * 100:.2f}%")

## STEP 7: EVALUASI & VISUALISASI CONFUSION MATRIX

In [ ]:
# Visualisasi komparasi akurasi
names = list(results.keys())
accuracies = [results[n]["acc"] * 100 for n in names]

plt.figure(figsize=(8, 4))
bars = plt.bar(names, accuracies, color=["#38bdf8", "#10b981", "#818cf8"][:len(names)])
plt.title("Perbandingan Akurasi Model Klasifikasi Sentimen", fontsize=14, fontweight="bold")
plt.ylabel("Akurasi (%)")
plt.ylim(50, 100)
for b in bars:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2., h + 1, f"{h:.2f}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## STEP 8: LIVE SENTIMENT PREDICTOR (INTERAKTIF)

In [ ]:
def test_opini_baru(teks):
    from src.lexicon_labeler import analyze_sentiment_detail
    cleaned = preprocess_text(teks)
    details = analyze_sentiment_detail(teks)
    print("=" * 60)
    print(f"Input Asli : "{teks}"")
    print(f"Cleaned    : "{cleaned}"")
    print(f"Sentimen   : {details["label"]} (Skor: {details["score"]})")
    if details["matched_positive_words"]:
        print(f"Kata Positif: {details["matched_positive_words"]}")
    if details["matched_negative_words"]:
        print(f"Kata Negatif: {details["matched_negative_words"]}")
    if details["negation_flips"]:
        print(f"Negasi Ditemukan: {details["negation_flips"]}")
    print("=" * 60)

# Coba beberapa contoh:
test_opini_baru("Sangat setuju pak menteri, transaksi jadi simpel dan mantap!")
test_opini_baru("Jangan sampai koruptor cuci uang timbunan di kamar saat tukar uang baru!")
test_opini_baru("Bagaimana teknis penukaran uang baru di bank pelosok desa?")